#Codice

In [1]:
!pip install simpy

In [2]:
import simpy
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [11]:
def run_ddos_sim(SIM_TIME, T_0, PROC_TIME, ATTACK_RATE, A, L, dist_type,
                 norm_mean, norm_std, uni_min, uni_max, weibull_shape, weibull_scale):

    # --- Inizializzazione Ambiente e Variabili ---
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)

    time_points = []
    queue_lengths = []

    # Contatori per le metriche richieste
    total_generated = 0
    total_rejected = 0

    def monitor(env):
        while True:
            time_points.append(env.now)
            queue_lengths.append(len(server.queue))
            yield env.timeout(0.01)

    env.process(monitor(env))

    def handle_request(env):
        nonlocal total_generated, total_rejected

        # 1. Il pacchetto è stato generato
        total_generated += 1

        # Controllo capacità coda
        if len(server.queue) < L:
            with server.request() as req:
                yield req
                yield env.timeout(PROC_TIME)
        else:
            # Drop del pacchetto (coda satura)
            yield env.timeout(0)
            total_rejected += 1

    # Generazione di richieste

    def attacker(env, attacker_id):
        delay = 0.0
        if dist_type == 'Gaussiana':
            delay = max(0.0, np.random.normal(norm_mean, norm_std))
        elif dist_type == 'Uniforme':
            t_min, t_max = min(uni_min, uni_max), max(uni_min, uni_max)
            delay = np.random.uniform(t_min, t_max)
        elif dist_type == 'Weibull':
            delay = weibull_scale * np.random.weibull(weibull_shape)

        start_delay = T_0 + delay
        yield env.timeout(start_delay)

        while True:
            env.process(handle_request(env))
            yield env.timeout(np.random.exponential(ATTACK_RATE))

    for i in range(A):
        env.process(attacker(env, i))

    # Esecuzione
    env.run(until=SIM_TIME)

    # --- Grafico ---
    plt.figure(figsize=(12, 6))
    plt.plot(time_points, queue_lengths, color='darkblue', linewidth=2, label='Richieste in coda')
    plt.axhline(y=L, color='red', linestyle='--', linewidth=2, label='Capacità massima (L)')

    if dist_type == 'Gaussiana':
        dist_info = f"Gaussiana (μ={norm_mean}, σ={norm_std})"
    elif dist_type == 'Uniforme':
        dist_info = f"Uniforme (Tmin={uni_min}, Tmax={uni_max})"
    else:
        dist_info = f"Weibull (forma={weibull_shape}, scala={weibull_scale})"

    plt.title(f"N={A}| t_r={ATTACK_RATE}s | L={L} | t_s={PROC_TIME}s\nRitardo: {dist_info}", fontsize=14)
    plt.xlabel("Tempo di simulazione (s)", fontsize=12)
    plt.ylabel("Numero di elementi in coda", fontsize=12)
    plt.fill_between(time_points, queue_lengths, color='blue', alpha=0.2)
    plt.ylim(-2, L + (L * 0.1))
    plt.xlim(0, SIM_TIME)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(loc='lower right')
    plt.show()

    # --- Stampa delle Metriche Finali ---
    drop_rate = (total_rejected / total_generated * 100) if total_generated > 0 else 0.0
    total_processed = total_generated - total_rejected

    print("-" * 50)
    print(f"📊 REPORT FINALE SIMULAZIONE (Tempo: {SIM_TIME}s)")
    print("-" * 50)
    print(f"🔹 Pacchetti Totali Generati   : {total_generated}")
    print(f"✅ Pacchetti Processati        : {total_processed}")
    print(f"❌ Pacchetti Rigettati         : {total_rejected} ({drop_rate:.1f}%)")
    print("-" * 50)

In [12]:
# --- Funzioni Helper per creare Slider lunghi + Casella di Input ---
style = {'description_width': '180px'}
slider_layout = widgets.Layout(width='450px') # Slider più lunghi
box_layout = widgets.Layout(width='100px')    # Casella di input

def make_int_param(desc, val, vmin, vmax, vstep):
    slider = widgets.IntSlider(value=val, min=vmin, max=vmax, step=vstep,
                               description=desc, style=style, layout=slider_layout,
                               continuous_update=False, readout=False)
    num_box = widgets.BoundedIntText(value=val, min=vmin, max=vmax, step=vstep, layout=box_layout)
    widgets.link((slider, 'value'), (num_box, 'value'))
    return widgets.HBox([slider, num_box]), slider

def make_float_param(desc, val, vmin, vmax, vstep):
    slider = widgets.FloatSlider(value=val, min=vmin, max=vmax, step=vstep,
                                 description=desc, style=style, layout=slider_layout,
                                 continuous_update=False, readout=False)
    num_box = widgets.BoundedFloatText(value=val, min=vmin, max=vmax, step=vstep, layout=box_layout)
    widgets.link((slider, 'value'), (num_box, 'value'))
    return widgets.HBox([slider, num_box]), slider

# --- Creazione Widget ---
hb_SIM_TIME, w_SIM_TIME = make_float_param('Tempo Simulazione T_end:', 40.0, 10.0, 200.0, 5.0)
hb_PROC_TIME, w_PROC_TIME = make_float_param('Tempo Servizio t_s:', 0.01, 0.001, 0.1, 0.001)
hb_L, w_L = make_int_param('Lunghezza coda L:', 100, 10, 1000, 10)

hb_A, w_A = make_int_param('Numero Attaccanti N:', 100, 1, 500, 5)
hb_ATTACK_RATE, w_ATTACK_RATE = make_float_param('Tempo medio tra richieste t_r:', 0.1, 0.01, 1.0, 0.01)
hb_T_0, w_T_0 = make_float_param('Inizio Attacco T_0:', 1.0, 0.0, 50.0, 1.0)

w_dist = widgets.Dropdown(options=['Gaussiana', 'Uniforme', 'Weibull'], value='Gaussiana',
                          description='Distribuzione Ritardo:', style=style, layout=widgets.Layout(width='350px'))

# Widget per le distribuzioni
hb_norm_mean, w_norm_mean = make_float_param('Media:', 5.0, 0.0, 20.0, 0.5)
hb_norm_std, w_norm_std = make_float_param('Dev. Std:', 2.0, 0.1, 10.0, 0.1)
box_gauss = widgets.VBox([hb_norm_mean, hb_norm_std])

hb_uni_min, w_uni_min = make_float_param('T min:', 0.0, 0.0, 20.0, 1.0)
hb_uni_max, w_uni_max = make_float_param('T max:', 10.0, 1.0, 30.0, 1.0)
box_uni = widgets.VBox([hb_uni_min, hb_uni_max])

hb_weibull_shape, w_weibull_shape = make_float_param('Forma:', 1.5, 0.1, 5.0, 0.1)
hb_weibull_scale, w_weibull_scale = make_float_param('Scala:', 5.0, 0.5, 30.0, 0.5)
box_weibull = widgets.VBox([hb_weibull_shape, hb_weibull_scale])

# Gestione visualizzazione dinamica
def update_ui(*args):
    box_gauss.layout.display = 'none'
    box_uni.layout.display = 'none'
    box_weibull.layout.display = 'none'

    if w_dist.value == 'Gaussiana':
        box_gauss.layout.display = 'flex'
    elif w_dist.value == 'Uniforme':
        box_uni.layout.display = 'flex'
    elif w_dist.value == 'Weibull':
        box_weibull.layout.display = 'flex'

w_dist.observe(update_ui, 'value')
update_ui()

# Costruzione interfaccia
ui = widgets.VBox([
    widgets.HTML("<b>Configurazione Globale e Server</b>"),
    hb_SIM_TIME, hb_PROC_TIME, hb_L,
    widgets.HTML("<hr><b>Configurazione Attaccanti</b>"),
    hb_A, hb_ATTACK_RATE, hb_T_0,
    widgets.HTML("<hr><b>Incertezza Attivazione (Ritardo da T_0)</b>"),
    w_dist,
    box_gauss, box_uni, box_weibull
])

# Collegamento output
out = widgets.interactive_output(run_ddos_sim, {
    'SIM_TIME': w_SIM_TIME, 'T_0': w_T_0,
    'PROC_TIME': w_PROC_TIME, 'ATTACK_RATE': w_ATTACK_RATE,
    'A': w_A, 'L': w_L, 'dist_type': w_dist,
    'norm_mean': w_norm_mean, 'norm_std': w_norm_std,
    'uni_min': w_uni_min, 'uni_max': w_uni_max,
    'weibull_shape': w_weibull_shape, 'weibull_scale': w_weibull_scale
})



# Simulazione

In [13]:
display(ui, out)

Output()